# Try TabICL on your own dataset

Edit the **CONFIG** cell below. Run all. Done.

TabICL is a pretrained foundation model for tables. No fine-tuning, no hyperparameter search — it does in-context learning from your training rows at inference time.

**You need:**
1. A table — `.csv`, `.tsv`, or `.parquet`. Local path or `s3://` URI both fine.
2. The name of the column to predict. Binary or multiclass classification.
3. `uv sync` once in this repo to pull TabICL + s3fs.

**No dataset handy?** Leave `DATA = "sample"` (the default) — the notebook will load sklearn's breast-cancer dataset (569 rows, 30 features, binary target). Lets you run the whole thing end-to-end in ~30 seconds.

## Config

Change these. Everything below runs without further edits.

In [ ]:
DATA = "sample"                          # "sample" → sklearn breast-cancer demo · or local path · or s3://bucket/key
TARGET = "target"                        # column to predict ("target" matches the sample dataset)

TEST_SIZE = 0.2                          # fraction held out for evaluation
SEED = 42                                # random seed for the split
MAX_TRAIN_ROWS = 10_000                  # cap — TabICL slows past ~10k rows on CPU

## Setup

`KMP_DUPLICATE_LIB_OK` silences a harmless libomp clash between torch and xgboost on macOS.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    matthews_corrcoef,
    roc_auc_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)
from tabicl import TabICLClassifier

## Load

If `DATA == "sample"`, sklearn's breast-cancer dataset is loaded (binary classification, 569 rows × 30 numeric features, target column named `target`).

Otherwise the file extension picks the reader. S3 URIs work because `s3fs` is in the project deps.

In [ ]:
if DATA == "sample":
    bunch = load_breast_cancer(as_frame=True)
    df = bunch.frame
    print("Loaded sklearn breast-cancer demo dataset.")
else:
    suffix = Path(DATA).suffix.lower()
    if suffix == ".parquet":
        df = pd.read_parquet(DATA)
    elif suffix in {".csv", ".tsv", ".txt"}:
        df = pd.read_csv(DATA, sep="\t" if suffix == ".tsv" else ",")
    else:
        raise ValueError(f"Unsupported extension {suffix!r}. Use .csv, .tsv, or .parquet.")

print(f"rows={len(df):,}  cols={df.shape[1]}")
df.head()

## Inspect target

Sanity check the class balance before fitting. Heavy imbalance is fine — TabICL handles it without weighting.

In [ ]:
y_full = df[TARGET]
X_full = df.drop(columns=[TARGET])

mask = y_full.notna()
X_full, y_full = X_full[mask], y_full[mask]

print(f"classes: {sorted(y_full.unique().tolist())}")
y_full.value_counts(normalize=True).round(3)

## Split

Stratified split keeps class proportions in both folds. Train is capped at `MAX_TRAIN_ROWS` because TabICL passes the whole train set as context — runtime is roughly linear in train size.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=TEST_SIZE, random_state=SEED, stratify=y_full,
)

if len(X_train) > MAX_TRAIN_ROWS:
    print(f"Subsampling train from {len(X_train):,} to {MAX_TRAIN_ROWS:,} rows.")
    X_train = X_train.sample(MAX_TRAIN_ROWS, random_state=SEED)
    y_train = y_train.loc[X_train.index]

print(f"train={len(X_train):,}  test={len(X_test):,}")

## Fit

One call. No grid search, no class weights, no early stopping. First run downloads the pretrained checkpoint from Hugging Face (~100 MB) — subsequent runs are cached.

In [ ]:
clf = TabICLClassifier(random_state=SEED, verbose=False, n_jobs=-1)
clf.fit(X_train, y_train)

## Predict + score

Accuracy is meaningless under imbalance — MCC and ROC-AUC are the numbers to watch for binary tasks.

In [ ]:
y_pred = clf.predict(X_test)

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"MCC      : {matthews_corrcoef(y_test, y_pred):.4f}")
if y_full.nunique() == 2:
    proba = clf.predict_proba(X_test)[:, 1]
    print(f"ROC-AUC  : {roc_auc_score(y_test, proba):.4f}")

## Confusion matrix

Rows are truth, columns are predictions. Off-diagonal cells are the errors.

In [ ]:
pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=[f"true_{c}" for c in clf.classes_],
    columns=[f"pred_{c}" for c in clf.classes_],
)

## Per-class report

Precision and recall per class. The minority-class row is usually the one that matters.

In [ ]:
print(classification_report(y_test, y_pred, digits=4))

## Next steps

- Tune the **threshold** on `proba` if MCC matters more than the default 0.5 cut.
- Add **feature engineering** — TabICL benefits from log transforms and ratios just like XGBoost did on the domestic-building task.
- Compare against your current model on the same split. Same `SEED` reproduces the split exactly.
- Swap `DATA` for your own table to see whether TabICL beats whatever you're using today.